In [2]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/dataset
!unzip -qo "/content/drive/MyDrive/deepsea/dataset.zip" -d /content/

Mounted at /content/drive


In [3]:
import os
print(os.listdir("/content/dataset"))

['instance_version', 'scripts', 'README.txt', 'original_data', 'material_version']


In [4]:
!nvidia-smi

Mon Nov 23 15:04:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!ls /content/dataset
!ls /content/dataset/dataset

instance_version  material_version  original_data  README.txt  scripts
ls: cannot access '/content/dataset/dataset': No such file or directory


In [6]:
!find /content/dataset -name "*.json"

Streaming output truncated to the last 5000 lines.
/content/dataset/original_data/annotations/vid_000119_frame0000013.jpg.json
/content/dataset/original_data/annotations/vid_000345_frame0000013.jpg.json
/content/dataset/original_data/annotations/vid_000282_frame0000012.jpg.json
/content/dataset/original_data/annotations/vid_000152_frame0000013.jpg.json
/content/dataset/original_data/annotations/vid_000369_frame0000013.jpg.json
/content/dataset/original_data/annotations/vid_000075_frame0000038.jpg.json
/content/dataset/original_data/annotations/vid_000276_frame0000025.jpg.json
/content/dataset/original_data/annotations/vid_000080_frame0000079.jpg.json
/content/dataset/original_data/annotations/vid_000048_frame0000012.jpg.json
/content/dataset/original_data/annotations/vid_000159_frame0000025.jpg.json
/content/dataset/original_data/annotations/vid_000040_frame0000046.jpg.json
/content/dataset/original_data/annotations/vid_000264_frame0000020.jpg.json
/content/dataset/original_data/annota

In [8]:
import json

chemin = "/content/dataset/dataset/instance_version/instances_train_trashcan.json"
with open(chemin) as f:
    data = json.load(f)

FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/dataset/instance_version/instances_train_trashcan.json'

In [ ]:
data.keys()

In [ ]:
len(data["images"])

In [ ]:
data["images"][0]

In [ ]:
data["categories"]

Une annotation, c’est une réponse écrite par un humain

Imagine que tu veux apprendre à un enfant à reconnaître un sac plastique sur une photo. Tu lui montres la photo, tu prends un stylo et tu entoures le sac en disant : « ça, c’est un sac plastique ».

Ce que tu viens de faire, c’est une annotation. Une personne a regardé chaque photo du dataset et a entouré à la main chaque objet (ROV, poisson, déchet...) en notant sa classe.

Ce qu’il y a dans une annotation

Reprenons la première annotation de ton fichier :

Champ	Ce que ça veut dire
image_id: 1	« Sur la photo n°1... »
category_id: 1	« ...il y a un ROV (classe 1)... »
segmentation: [415, 270, 415.9, 263, ...]

In [ ]:
data["annotations"][0]

Annotation (liste de points)  →  Masque (tableau de 0 et de 1)                                                    Annotation = le contour d’un objet dessiné par un humain. Masque = la même information transformée en pixels 0/1, pour que le modèle puisse apprendre avec.

In [ ]:
id_vers_nom = {c["id"]: c["name"] for c in data["categories"]}
id_vers_nom

In [ ]:
ids_dechets = [i for i, nom in id_vers_nom.items() if nom.startswith("trash_")]
print(len(ids_dechets))
print(ids_dechets)

rov 2633 veut dire 2633 annotations de ROV, pas 2633 images.

In [ ]:
from collections import Counter

compteur = Counter(id_vers_nom[a["category_id"]] for a in data["annotations"])
for nom, n in compteur.most_common():
    print(nom, n)

In [ ]:
import re

videos = {re.search(r"(vid_\d+)", img["file_name"]).group(1) for img in data["images"]}
print("Vidéos dans le train :", len(videos))

310 vidéos. Les 6065 images sont des frames extraites de 310 vidéos, donc plusieurs images viennent de la même vidéo

Étape 5 : construire un masque pour une image

In [ ]:
from pycocotools.coco import COCO
import numpy as np
coco = COCO(chemin)

cherche la première annotation dont la classe est un déchet. Résultat : image 2, trash_bag.

In [ ]:
ann= next(a for a in data["annotations"]if a ["category_id"] in ids_dechets)
img_id=ann["image_id"]
print(img_id,id_vers_nom[ann["category_id"]])

In [ ]:
info=coco.imgs[img_id]
mask=np.zeros((info["height"],info["width"]),dtype=np.uint8)
ann_ids=coco.getAnnIds(imgIds=img_id , catIds= ids_dechets, iscrowd=None)
for a in coco.loadAnns(ann_ids):
    mask = np.maximum(mask, coco.annToMask(a))
print(mask.shape,np.unique(mask),mask.sum())


mask.sum() → 599 : le nombre de pixels de déchet
np.unique(mask) → [0 1] : les valeurs présentes
mask.shape → (270, 480) : la taille

la vérification visuelle

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

racine = "dataset/dataset/instance_version/train/"
photo = Image.open(racine + info["file_name"])

fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(photo);                  ax[0].set_title("Photo")
ax[1].imshow(mask, cmap="gray");      ax[1].set_title("Masque (1 = déchet)")
ax[2].imshow(photo)
ax[2].imshow(np.ma.masked_where(mask == 0, mask), alpha=0.6, cmap="autumn")
ax[2].set_title("Superposition")
plt.show()

une fonction de masque réutilisable

In [ ]:
def construire_masque(img_id):
    info = coco.imgs[img_id]
    mask = np.zeros((info["height"], info["width"]), dtype=np.uint8)
    ann_ids = coco.getAnnIds(imgIds=img_id)
    for ann in coco.loadAnns(ann_ids):
        nom_classe = coco.cats[ann["category_id"]]["name"]
        if nom_classe.startswith("trash_"):
            mask = np.maximum(mask, coco.annToMask(ann))
    return info, mask

trouver un exemple de chaque cas

In [ ]:
# images contenant un ROV, un animal, ou aucun déchet
cas = {"rov": None, "animal": None, "sans_dechet": None}

for img_id in coco.imgs:
    noms = [coco.cats[a["category_id"]]["name"]
            for a in coco.loadAnns(coco.getAnnIds(imgIds=img_id))]
    a_dechet = any(n.startswith("trash_") for n in noms)

    if cas["rov"] is None and "rov" in noms and not a_dechet:
        cas["rov"] = img_id
    if cas["animal"] is None and any(n.startswith("animal_") for n in noms) and not a_dechet:
        cas["animal"] = img_id
    if cas["sans_dechet"] is None and len(noms) > 0 and not a_dechet:
        cas["sans_dechet"] = img_id

    if all(v is not None for v in cas.values()):
        break

print(cas)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 5))

for i, (nom, img_id) in enumerate(cas.items()):
    info, mask = construire_masque(img_id)
    photo = Image.open(racine + info["file_name"])
    ax[i].imshow(photo)
    ax[i].imshow(np.ma.masked_where(mask == 0, mask), alpha=0.6, cmap="autumn")
    ax[i].set_title(f"{nom} (img {img_id}) : {mask.sum()} px")
    print(nom, img_id, "pixels déchet =", mask.sum())

plt.show()

statistiques sur tout le train

In [ ]:
import pandas as pd

lignes = []
for img_id in coco.imgs:
    info, mask = construire_masque(img_id)
    lignes.append({
        "img_id": img_id,
        "file_name": info["file_name"],
        "pixels_dechet": int(mask.sum()),
        "pixels_total": mask.size,
    })

df = pd.DataFrame(lignes)
df["proportion"] = df["pixels_dechet"] / df["pixels_total"]
df["a_dechet"] = df["pixels_dechet"] > 0

print("Nombre d'images :", len(df))
print("Avec déchet :", df["a_dechet"].sum())
print("Sans déchet :", (~df["a_dechet"]).sum())
print("Proportion moyenne de déchet (toutes images) : {:.2%}".format(df["proportion"].mean()))
print("Proportion moyenne (images avec déchet seulement) : {:.2%}".format(df.loc[df["a_dechet"], "proportion"].mean()))
print(df["proportion"].describe())

In [ ]:
ligne_max = df.loc[df["proportion"].idxmax()]
print(ligne_max)

info, mask = construire_masque(int(ligne_max["img_id"]))
photo = Image.open(racine + info["file_name"])
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
ax[0].imshow(photo)
ax[1].imshow(photo)
ax[1].imshow(np.ma.masked_where(mask == 0, mask), alpha=0.6, cmap="autumn")
plt.show()

ÉTAPE 7 : extraire l'identifiant de vidéo depuis le nom de fichier


## Étape 7 : fusionner train et val officiels, puis extraire la vidéo de chaque image
Le split officiel n'est pas séparé par vidéo (113 des 115 vidéos du val apparaissent aussi dans le train). On fusionne donc les deux fichiers (7 212 images) et on construit un seul tableau, avec la colonne `source` pour retrouver le dossier de chaque image.

Le rôle de l’étape 7 en une phrase : retrouver, pour chaque image, la vidéo dont elle vient, afin de pouvoir ranger toutes les images d’une même vidéo dans le même groupe (train, validation ou test) à l’étape 8.

Cette étape ne change ni les images, ni les masques, ni le modèle : elle ajoute seulement une étiquette de rangement. C’est la condition pour que l’évaluation soit honnête (pas de fuite de données).

In [ ]:
import os, numpy as np, pandas as pd
from pycocotools.coco import COCO

base = "dataset/dataset/instance_version/"
print(os.listdir(base))   # vérifier les noms exacts du json val et du dossier val

sources = {
    "train": (base + "instances_train_trashcan.json", base + "train/"),
    "val":   (base + "instances_val_trashcan.json",   base + "val/"),
}
cocos = {s: COCO(chemin) for s, (chemin, _) in sources.items()}
dossiers = {s: dossier for s, (_, dossier) in sources.items()}

In [ ]:
def construire_masque_complet(source, img_id):
    c = cocos[source]
    info = c.imgs[img_id]
    mask = np.zeros((info["height"], info["width"]), dtype=np.uint8)
    for ann in c.loadAnns(c.getAnnIds(imgIds=img_id)):
        if c.cats[ann["category_id"]]["name"].startswith("trash_"):
            mask = np.maximum(mask, c.annToMask(ann))
    return info, mask

In [ ]:
lignes = []
for s in sources:
    for img_id in cocos[s].imgs:
        info, mask = construire_masque_complet(s, img_id)
        lignes.append({"source": s, "img_id": img_id,
                       "file_name": info["file_name"],
                       "pixels_dechet": int(mask.sum()),
                       "pixels_total": mask.size})

df = pd.DataFrame(lignes)
df["proportion"] = df["pixels_dechet"] / df["pixels_total"]
df["a_dechet"] = df["pixels_dechet"] > 0
df["video"] = df["file_name"].str.extract(r"(vid_\d+)", expand=False)

assert df["video"].isna().sum() == 0
print(len(df), df["video"].nunique())              # attendu : 7212 312
print(df["source"].value_counts())                 # attendu : train 6065, val 1147
print("Doublons :", df["file_name"].duplicated().sum())   # attendu : 0

In [ ]:
videos = np.array(sorted(df["video"].unique()))
rng = np.random.default_rng(42)      # graine 42 : split reproductible
rng.shuffle(videos)

n = len(videos)
n_train = int(0.70 * n)
n_val = int(0.15 * n)
videos_train = videos[:n_train]
videos_val = videos[n_train:n_train + n_val]
videos_test = videos[n_train + n_val:]
print("Vidéos train/val/test :", len(videos_train), len(videos_val), len(videos_test))

In [ ]:
s_train, s_val, s_test = set(videos_train), set(videos_val), set(videos_test)
assert not (s_train & s_val), "Vidéos à la fois dans train et val"
assert not (s_train & s_test), "Vidéos à la fois dans train et test"
assert not (s_val & s_test), "Vidéos à la fois dans val et test"
assert len(s_train) + len(s_val) + len(s_test) == n, "Il manque des vidéos"

groupe_de = {}
for v in videos_train: groupe_de[v] = "train"
for v in videos_val:   groupe_de[v] = "val"
for v in videos_test:  groupe_de[v] = "test"
df["split"] = df["video"].map(groupe_de)
assert df["split"].isna().sum() == 0
print("Aucune fuite : les 3 groupes sont séparés.")

In [ ]:
resume = df.groupby("split").agg(images=("img_id", "count"),
                                 videos=("video", "nunique"))
resume["part_images_%"] = (100 * resume["images"] / len(df)).round(1)
print(resume.loc[["train", "val", "test"]])

equilibre = df.groupby("split").agg(
    part_avec_dechet=("a_dechet", "mean"),
    proportion_moyenne=("proportion", "mean"),
    proportion_mediane=("proportion", "median"),
).loc[["train", "val", "test"]]
print(equilibre.round(4))

In [ ]:
df[["source", "img_id", "file_name", "video", "split"]].to_csv("split_liste.csv", index=False)
print(df["split"].value_counts())

In [ ]:
print(len(df), df["video"].nunique())      # attendu : 7212 et 312

print(df["file_name"].duplicated().sum())  # attendu : 0
print(df["split"].value_counts())          # nouveau split sur 7212 images